# LLM-Driven Bayesian Modeling Tool — Spiritual Resilience Training (AIE Pilot)
### Adapted Tutorial — Religious Training For Spiritual Resilience (Army, FY2027)

**Using Hubbard's Applied Information Economics (AIE) + Large Language Models**

---

## Purpose of This Adaptation

This notebook reuses the AIE + Bayesian + LLM workflow from the reliability tutorial,
but applies it to a **non-equipment** decision: how much to fund a 2027 program that
trains 4,000 soldiers in spiritual resilience. The original notebook models *failures
of pumps*; here we model *failures of soldiers* — where 'failure' means moral/spiritual
injury leading to career attrition, suicidal ideation, or suicide attempt.

> **Status:** Fully rewired. Sections 5–7 and 9 use a Beta-Binomial / risk-pool model
> on the three resilience beliefs (B1, B2, B3) with calibrated Normal priors on the
> training uplift. Independence between the three beliefs is assumed — flagged as
> VoI item #1 in the *Measures to Develop* cell after Section 3.

---

### Tutorial Sections (unchanged structure)

| # | Section | AIE Step |
|---|---------|----------|
| 1 | Environment Setup | — |
| 2 | Background: AIE + Bayesian Concepts | Framing |
| 3 | Step 1 — Define the Decision & Uncertainty | AIE Step 1 |
| 4 | Step 2 — LLM Problem Parser → JSON | AIE Step 2 |
| 5 | Step 3 — Value of Information (VoI) Ranking | AIE Step 3 |
| 6 | Step 4 — Calibrated Priors (Two Paths) | AIE Step 4 |
| 7 | Step 5 — Bayesian Update (Likelihood + Posterior) | AIE Step 5 |
| 8 | Step 6 — Chained Bayesian DAG Visualization | AIE Step 5 |
| 9 | Step 7 — Monte Carlo Simulation & Decision | AIE Step 5 |
|10 | Your Turn — Participant Exercise | All Steps |


---
## Section 1 — Environment Setup

Run this cell once to install required packages.  
On **Google Colab**: this works immediately with no local configuration.

In [ ]:
# Install required packages (safe to re-run)
# Uses uv when available (required for uv-managed .venv); falls back to pip (Colab etc.)
import subprocess, sys, shutil, os, importlib.util

packages = [
    "numpy", "scipy", "pandas", "matplotlib", "seaborn",
    "networkx",       # DAG visualization
    "openai",         # LLM API (OpenAI-compatible; swap for any provider)
    "pymc",           # Bayesian inference
    "arviz",          # Posterior diagnostics
    "ipywidgets",     # Interactive sliders
]

# Only install packages that are actually missing
missing = [pkg for pkg in packages if importlib.util.find_spec(pkg) is None]
if missing:
    print(f'Installing missing packages: {missing}')
    uv_candidates = [
        shutil.which('uv'),
        os.path.join(os.path.expanduser('~'), '.local', 'bin', 'uv.exe'),
        os.path.join(os.path.expanduser('~'), '.local', 'bin', 'uv'),
    ]
    uv_exe = next((p for p in uv_candidates if p and os.path.isfile(p)), None)
    if uv_exe:
        subprocess.check_call([uv_exe, 'pip', 'install', '--python', sys.executable] + missing)
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
else:
    print('All packages already installed — skipping.')

print('All packages ready.')

In [ ]:
# Core imports
import os, json, textwrap
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import pymc as pm
import arviz as az
import ipywidgets as widgets
from IPython.display import display, Markdown, JSON

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")
print("Environment ready.")

---
## Section 2 — Background: AIE + Bayesian Concepts

### Doug Hubbard's Applied Information Economics (AIE) — 5 Steps

| Step | Action | Bayesian Equivalent |
|------|--------|---------------------|
| 1 | Define the decision and what is uncertain | Identify random variables |
| 2 | Determine what measurements matter | VoI analysis |
| 3 | Measure highest-value items | Collect evidence / data |
| 4 | Use information to update beliefs | Bayesian update: prior × likelihood → posterior |
| 5 | Make the decision | Expected utility / loss function |

### The Two Persistent Barriers

**Barrier 1 — Distribution Quality**  
Engineers default to uniform or normal distributions when the physics, failure modes, and data suggest something very different (Weibull, Beta, Lognormal, etc.).

**Barrier 2 — Parameter Mapping**  
Even engineers who understand Bayes struggle to map their specific problem into:  
$$P(\theta \mid \text{data}) \propto P(\text{data} \mid \theta) \cdot P(\theta)$$

**This tool solves both** by using an LLM to parse the problem description, select the appropriate distributions, and generate the full JSON model structure automatically.

### Worked Example Throughout This Tutorial

> **Problem**: A centrifugal pump at a water treatment facility has experienced 3 failures in 24 months of operation.  
> Maintenance cost per intervention: \$4,200. Unplanned failure cost: \$18,000.  
> **Decision**: Should we implement a 6-month preventive maintenance (PM) schedule, or continue run-to-failure?

---
## Section 3 — Step 1: Define the Decision & Uncertainties

We frame the **2027 funding decision** for the *Religious Training For Spiritual
Resilience* (RTSR) program before invoking the LLM.

### Background facts (from the problem statement)

- Target population: **4,000 soldiers** to train in 2027.
- Per-soldier program cost: **$15 meal + $20 materials + $1.50 instructor share** ≈ **$36.50/soldier**,
  driven mostly by an instructor block at **$6,000** amortized across the cohort.
  Total program ≈ **$150,000**.
- Funded only to **$50,000** ⇒ shortfall = **$100,000** ⇒ only ~33% of soldiers trained.
- Baseline survey of the 4,000 soldiers — fraction holding each resilience belief:
  - **40%** are *resolved in their spiritual beliefs* (B1)
  - **60%** can *identify a cause greater than self* worth suffering for (B2)
  - **75%** *believe in a higher power that enables them to overcome* (B3)
- Risk tier as a function of how many of {B1, B2, B3} a soldier holds:
  - **3 of 3** → resilient (low risk)
  - **2 of 3** → moderate risk of moral/spiritual injury → career attrition or suicidal ideation
  - **0–1 of 3** → high risk → attrition and/or suicide attempt
- Training effects (point estimates, to be turned into calibrated ranges):
  - +30% likelihood to *seek help / become spiritually resilient* (general)
  - +50% likelihood among those specifically trained in spiritual resilience
- Known rates per 100,000 soldiers (placeholders; user must supply current Army values):
  - Suicide rate = **X / 100,000**
  - Suicidal ideation or attempt rate = **Y / 100,000**
- Intangibles **not yet quantified**: mission distraction, shortened career longevity,
  unit cohesion loss, family/dependent impact.

### Template: Decision Frame


In [ ]:
# ── DECISION FRAME ─────────────────────────────────────────────────────────────
# Religious Training For Spiritual Resilience (RTSR) — FY2027 funding decision.
# Replace placeholders X, Y with current Army figures when available.

X_SUICIDE_PER_100K  = 25.0    # PLACEHOLDER — replace with current Army suicide rate
Y_IDEATION_PER_100K = 800.0   # PLACEHOLDER — replace with ideation/attempt rate

decision_frame = {
    "problem_description": """
        The U.S. Army is launching a 2027 program — Religious Training For Spiritual
        Resilience (RTSR) — to train 4,000 soldiers to resolve their core beliefs and
        select supportive practices and faith communities. Full funding is $150,000
        ($15 meal + $20 materials per soldier, plus instructor blocks at $6,000).

        Only $50,000 is currently committed, a $100,000 shortfall, so only ~33% of
        soldiers (≈1,320) would be trained.

        A baseline survey of the 4,000 indicates: 40% are resolved in spiritual
        beliefs (B1), 60% can name a cause greater than self worth sacrificing for
        (B2), and 75% believe in a higher power that helps them overcome (B3).
        Soldiers holding all three are most resilient. Two -> moderate risk of
        moral/spiritual injury (career attrition, suicidal ideation). One or zero ->
        high risk (attrition, suicide attempt).

        Training is estimated to make soldiers 30% more likely to seek help in
        general, and 50% more likely among those specifically trained in spiritual
        resilience. Suicide rate is X/100,000 and ideation/attempt rate is Y/100,000.
        Intangibles (mission distraction, career longevity, unit cohesion) are not
        yet quantified.

        Decision: should the program be fully funded ($150K) for all 4,000, partially
        funded ($50K, ~33% coverage), or expanded with stretch funding to deepen the
        intervention?
    """,
    "decision_options": [
        "Full funding $150K — train all 4,000 soldiers in 2027",
        "Partial funding $50K — train ~33% (current plan)",
        "No funding — cancel 2027 program",
    ],
    "objective": "Minimize expected total cost (program + injury/attrition/suicide-related cost) over FY2027",
    "time_horizon_months": 12,

    # ── Population & program economics ─────────────────────────────────────────
    "target_population": 4000,
    "program_cost_full_usd": 150_000,
    "program_cost_partial_usd": 50_000,
    "fraction_trained_partial": 0.33,
    "per_soldier_meal_usd": 15,
    "per_soldier_materials_usd": 20,
    "instructor_block_usd": 6_000,

    # ── Belief prevalences (Bernoulli priors per soldier, baseline) ────────────
    "p_belief_resolved_baseline": 0.40,   # B1
    "p_belief_cause_baseline":    0.60,   # B2
    "p_higher_power_baseline":    0.75,   # B3

    # ── Risk-tier mapping (count of beliefs held -> tier) ──────────────────────
    "tier_by_belief_count": {
        3: "resilient (low risk)",
        2: "moderate risk",
        1: "high risk",
        0: "high risk",
    },

    # ── Training effect estimates (point — convert to 90% CI later) ────────────
    "training_uplift_seek_help_general":     0.30,   # +30%
    "training_uplift_seek_help_rtsr_specific": 0.50, # +50%

    # ── Adverse-event base rates (per 100,000 soldiers) ────────────────────────
    "suicide_rate_per_100k": X_SUICIDE_PER_100K,
    "ideation_or_attempt_rate_per_100k": Y_IDEATION_PER_100K,

    # ── Unmodeled / intangible costs (qualitative for now) ─────────────────────
    "intangibles_not_yet_quantified": [
        "Mission distraction / readiness loss",
        "Shortened career longevity (separation, medical retirement)",
        "Unit cohesion / leader trust",
        "Family and dependent secondary impact",
        "Recruitment and replacement training cost per separated soldier",
    ],
}

display(Markdown("**Decision Frame Captured (RTSR FY2027):**"))
display(Markdown(f"- **Objective**: {decision_frame['objective']}"))
display(Markdown(f"- **Options**: {decision_frame['decision_options']}"))
display(Markdown(
    f"- **Population**: {decision_frame['target_population']:,} soldiers; "
    f"baseline beliefs B1/B2/B3 = "
    f"{decision_frame['p_belief_resolved_baseline']:.0%}/"
    f"{decision_frame['p_belief_cause_baseline']:.0%}/"
    f"{decision_frame['p_higher_power_baseline']:.0%}"
))
display(Markdown(
    f"- **Funding gap**: full ${decision_frame['program_cost_full_usd']:,} vs "
    f"partial ${decision_frame['program_cost_partial_usd']:,} "
    f"(coverage {decision_frame['fraction_trained_partial']:.0%})"
))


### Suggested Measures to Develop or Explore

Below are candidate **outcome measures** and **value drivers** that would let us
quantify the cost of under-resourcing RTSR in 2027. Items marked *(elicit)* are
good candidates for AIE calibrated estimates from chaplains, behavioral-health
officers, and command staff, since hard data is sparse.

**A. Belief / resilience metrics (the latent state)**
1. Joint distribution of {B1, B2, B3} per soldier (currently we have only marginals).
   Are the beliefs independent, or correlated? *(elicit)*
2. Pre/post-training shift in each marginal: ΔP(B1), ΔP(B2), ΔP(B3). *(measure via paired survey)*
3. Proportion of soldiers in each risk tier (3/2/1/0 of 3) before vs after training.

**B. Adverse-event rates (the cost surface)**
4. Suicide rate per 100k by risk tier (currently we have only the overall Army rate X).
5. Suicidal-ideation / attempt rate per 100k by risk tier (overall rate Y).
6. Career-attrition rate (separation, medical retirement) by risk tier.
7. Lost duty days per moral/spiritual injury event. *(elicit)*

**C. Training effectiveness (the intervention model)**
8. Calibrated 90% CI on the +30% 'seek help' uplift — cite source or elicit.
9. Calibrated 90% CI on the +50% RTSR-specific uplift.
10. Persistence of effect: half-life of the uplift in months. *(elicit)*
11. Dose-response: does a longer block (4 hr vs 2 hr) shift the uplift? *(experiment)*

**D. Economic conversion (the dollar-equivalent)**
12. Cost per soldier separation (recruitment + initial training replacement). *Army G-1 data.*
13. Cost per suicide attempt requiring hospitalization. *DHA / TRICARE data.*
14. Statistical value placed on a soldier life — use DoD or DOT VSL ($11–13M range).
15. Productivity / readiness cost of a moderate-risk soldier-month. *(elicit)*

**E. Intangibles to bound (not yet quantified)**
16. Mission distraction → unit performance score delta. *(elicit, then survey)*
17. Family / dependent secondary-injury rate per high-risk soldier. *(elicit)*
18. Trust in chain of command (Likert delta pre/post). *(survey)*

**F. Decision-relevant aggregate outputs (what the model must produce)**
19. Expected number of moral/spiritual injury events avoided per $1,000 invested.
20. Expected suicide attempts avoided per $1,000 invested.
21. P(at least one suicide avoided | full funding) − P(… | partial funding).
22. Risk-adjusted Net Present Value of the $100K shortfall.
23. Value of Information (VoI) for each measure above — i.e., which one most reduces
    decision uncertainty and is therefore worth measuring first.

Section 5 (VoI ranking) is the natural place to test whether items 4, 6, 8, or 12
are the highest-leverage measurements to fund as a precursor to the full program.


---
## Section 4 — Step 2: LLM Problem Parser → Structured JSON

We send the problem description to an LLM and receive a **structured JSON** that:
- Identifies all uncertain quantities (the measurement nodes)
- Assigns appropriate probability distribution families to each
- Specifies whether each node is a **prior**, **likelihood**, or **posterior**
- Explains *why* each distribution was chosen

### Configure Your LLM API Key

> **Note**: The cell below uses the OpenAI API. You can substitute any OpenAI-compatible endpoint  
> (Azure OpenAI, Groq, Ollama, xAI Grok, etc.) by changing `base_url` and `model`.

In [ ]:
# ── LLM CONFIGURATION ──────────────────────────────────────────────────────────
# This notebook uses a provider API endpoint. It does not call Copilot/Chat directly.
# Supported here: OpenAI and xAI Grok via the OpenAI-compatible client.


from openai import OpenAI

PROVIDER = os.environ.get("LLM_PROVIDER", "openai").lower()  # "openai" or "xai"

if PROVIDER == "openai":
    API_KEY  = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
    BASE_URL = "https://api.openai.com/v1"
    MODEL    = os.environ.get("LLM_MODEL", "gpt-4o-mini")
elif PROVIDER == "xai":
    API_KEY  = os.environ.get("XAI_API_KEY", "YOUR_KEY_HERE")
    BASE_URL = "https://api.x.ai/v1"
    MODEL    = os.environ.get("LLM_MODEL", "grok-4")
else:
    raise ValueError("Unsupported PROVIDER. Use 'openai' or 'xai'.")

# ── DIRECT-KEY OVERRIDE (uncomment + paste your key for quickest path) ─────────
# Get a key from https://platform.openai.com/api-keys (uses your existing $5 credit).
# Cost for this notebook with gpt-4o-mini: well under $0.01 per full run.
# API_KEY = "sk-PASTE-YOUR-OPENAI-KEY-HERE"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print(f"Provider: {PROVIDER}")
print(f"LLM configured: {MODEL} @ {BASE_URL}")
print(f"API_KEY loaded: length={len(API_KEY) if API_KEY else 0}, starts with {API_KEY[:4]!r}")

In [ ]:
# ── OPTIONAL API CONNECTION CHECK ─────────────────────────────────────────────
# Run this after the config cell to verify the provider, key, and model before the full parser.

print(f"Key length: {len(API_KEY) if API_KEY else 0}  | starts with: {API_KEY[:4] if API_KEY else ''!r}")

if (not API_KEY) or API_KEY.strip() in ("", "YOUR_KEY_HERE") or API_KEY.strip().startswith("YOUR_"):
    raise ValueError(
        "API key is still the placeholder. The kernel did not pick up your key. "
        "Easiest fix: open the LLM CONFIGURATION cell and set API_KEY = 'xai-...' directly (after the if/elif block), then re-run that cell and this one."
    )

try:
    test_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": "Reply with exactly: API connection ok"}
        ],
        temperature=0,
        max_tokens=10,
    )
    print(test_response.choices[0].message.content)
    print(f"API check passed for provider={PROVIDER}, model={MODEL}.")
except Exception as error:
    print(f"API check failed for provider={PROVIDER}, model={MODEL}: {type(error).__name__}: {error}")
    raise

In [ ]:
# ── SYSTEM PROMPT — Bayesian Model Engineer ────────────────────────────────────
SYSTEM_PROMPT = """
You are an expert Bayesian reliability engineer and statistician.
Given a natural-language reliability problem description, you must:

1. Identify all uncertain quantities (measurement nodes).
2. For each node, specify:
   - name: short identifier (snake_case)
   - description: plain-English explanation of what it represents
   - role: one of ["prior", "likelihood", "posterior", "decision_variable", "cost_node"]
   - distribution_family: e.g. Gamma, Weibull, Beta, Normal, Poisson, Exponential
   - distribution_parameters: dict of parameter names and initial values or ranges
   - justification: WHY this distribution family is correct for this node
   - data_source: "expert_estimate", "observed_data", or "derived"
   - depends_on: list of node names this node depends on (empty list if root node)
3. Identify the key Bayesian update chain as an ordered list of node names.
4. Return ONLY valid JSON. No markdown fences, no explanation outside the JSON.

JSON schema:
{
  "problem_summary": "string",
  "decision": "string",
  "measurement_nodes": [ { ...node fields... } ],
  "bayesian_update_chain": ["node_name", ...],
  "voi_candidates": ["node_name", ...],
  "model_notes": "string"
}
"""

def parse_problem_to_json(problem_text: str) -> dict:
    """Send problem description to LLM and return structured Bayesian JSON."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": problem_text}
        ],
        temperature=0.1,   # Low temperature for deterministic structured output
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

print("LLM parser function defined.")

In [ ]:
# ── CALL THE LLM ───────────────────────────────────────────────────────────────
# This sends the problem description and returns the structured Bayesian model.

bayesian_model_json = parse_problem_to_json(decision_frame["problem_description"])

# Pretty-print the result
print(json.dumps(bayesian_model_json, indent=2))

In [ ]:
# ── OFFLINE FALLBACK ───────────────────────────────────────────────────────────
# If you do not have an API key, use this pre-generated JSON for the pump example.
# Comment out the cell above and run this one instead.

OFFLINE_JSON = {
  "problem_summary": "Centrifugal pump with 3 failures in 24 months. Decide between 6-month PM schedule and run-to-failure over a 12-month horizon.",
  "decision": "Minimize expected total maintenance cost over 12 months",
  "measurement_nodes": [
    {
      "name": "failure_rate_lambda",
      "description": "True underlying failure rate of the pump (failures per month)",
      "role": "prior",
      "distribution_family": "Gamma",
      "distribution_parameters": {"alpha": 3.5, "beta": 12.0},
      "justification": "Gamma is the conjugate prior for a Poisson failure process. Alpha/beta seeded from industry MTBF range of 9-14 months (lambda ~ 0.07-0.11 failures/month).",
      "data_source": "expert_estimate",
      "depends_on": []
    },
    {
      "name": "observed_failures",
      "description": "Observed failure count: 3 failures in 24 months — the evidence term",
      "role": "likelihood",
      "distribution_family": "Poisson",
      "distribution_parameters": {"mu": "failure_rate_lambda * 24"},
      "justification": "Failures are discrete events; Poisson likelihood is standard for count data when events are independent and the rate is constant.",
      "data_source": "observed_data",
      "depends_on": ["failure_rate_lambda"]
    },
    {
      "name": "posterior_failure_rate",
      "description": "Updated belief about the true failure rate after observing 3 failures in 24 months",
      "role": "posterior",
      "distribution_family": "Gamma",
      "distribution_parameters": {"alpha": 6.5, "beta": 36.0},
      "justification": "Gamma-Poisson conjugacy: posterior Gamma(alpha + k, beta + t) = Gamma(3.5+3, 12+24) = Gamma(6.5, 36). Analytically exact.",
      "data_source": "derived",
      "depends_on": ["failure_rate_lambda", "observed_failures"]
    },
    {
      "name": "failures_next_12mo_pm",
      "description": "Predicted failures in next 12 months under PM schedule (reduced effective exposure)",
      "role": "decision_variable",
      "distribution_family": "Poisson",
      "distribution_parameters": {"mu": "posterior_failure_rate * 6"},
      "justification": "Under 6-month PM, each cycle's effective exposure is ~6 months before intervention resets wear accumulation. Expected failures = lambda * 6.",
      "data_source": "derived",
      "depends_on": ["posterior_failure_rate"]
    },
    {
      "name": "failures_next_12mo_rtf",
      "description": "Predicted failures in next 12 months under run-to-failure",
      "role": "decision_variable",
      "distribution_family": "Poisson",
      "distribution_parameters": {"mu": "posterior_failure_rate * 12"},
      "justification": "Full 12-month exposure with no preventive interventions.",
      "data_source": "derived",
      "depends_on": ["posterior_failure_rate"]
    },
    {
      "name": "total_cost_pm",
      "description": "Total cost under PM: 2 planned interventions + unplanned failures",
      "role": "cost_node",
      "distribution_family": "derived_from_simulation",
      "distribution_parameters": {"pm_interventions": 2, "cost_pm": 4200, "cost_failure": 18000},
      "justification": "2 planned PMs per year × $4,200 + expected unplanned failures × $18,000.",
      "data_source": "derived",
      "depends_on": ["failures_next_12mo_pm"]
    },
    {
      "name": "total_cost_rtf",
      "description": "Total cost under run-to-failure: all costs are unplanned failure costs",
      "role": "cost_node",
      "distribution_family": "derived_from_simulation",
      "distribution_parameters": {"cost_failure": 18000},
      "justification": "No planned interventions; all maintenance costs come from unplanned failures.",
      "data_source": "derived",
      "depends_on": ["failures_next_12mo_rtf"]
    }
  ],
  "bayesian_update_chain": [
    "failure_rate_lambda",
    "observed_failures",
    "posterior_failure_rate",
    "failures_next_12mo_pm",
    "failures_next_12mo_rtf",
    "total_cost_pm",
    "total_cost_rtf"
  ],
  "voi_candidates": ["failure_rate_lambda", "posterior_failure_rate"],
  "model_notes": "Gamma-Poisson conjugate model allows analytical posterior. Monte Carlo used for cost propagation. PM effectiveness assumed to reset wear accumulation at each intervention."
}

# Uncomment the line below to use offline fallback:
# bayesian_model_json = OFFLINE_JSON

print("Offline fallback JSON defined. Uncomment last line to use it.")

---
## Section 5 — Step 3: Value of Information (VoI) Ranking

Before collecting more data, AIE asks: **which uncertainty most affects the decision?**
We compute a simple Expected Value of Perfect Information (EVPI) proxy by perturbing
each input across its plausible range and measuring how much the **expected cost gap**
between funding options changes.

Inputs evaluated:
1. Training uplift on belief marginals (the +30% / +50% effects)
2. High-risk tier suicide multiplier (vs Army baseline X)
3. Cost per suicide event (VSL anchor)
4. Cost per moderate-risk separation / attrition event
5. Belief correlation structure (independence vs strong joint distribution)


In [ ]:
# ── VoI ANALYSIS (RTSR) ────────────────────────────────────────────────────────
# Quick sensitivity scan: for each uncertainty, vary it across a low/high range
# and recompute the expected cost difference between FULL and PARTIAL funding.
# The larger the swing, the higher the VoI of measuring it precisely.

import numpy as np
import pandas as pd

DF = decision_frame  # alias

def expected_cost(option: str, *,
                  uplift_general=0.30,
                  uplift_rtsr=0.50,
                  high_risk_suicide_mult=4.0,
                  cost_per_suicide=11_000_000,
                  cost_per_attempt=50_000,
                  cost_per_separation=85_000,
                  belief_corr=0.0) -> float:
    """Return expected total cost (program + adverse-event) for a funding option."""
    N = DF["target_population"]
    p1 = DF["p_belief_resolved_baseline"]
    p2 = DF["p_belief_cause_baseline"]
    p3 = DF["p_higher_power_baseline"]

    if option == "full":
        prog_cost = DF["program_cost_full_usd"]
        frac_trained = 1.0
    elif option == "partial":
        prog_cost = DF["program_cost_partial_usd"]
        frac_trained = DF["fraction_trained_partial"]
    else:
        prog_cost = 0.0
        frac_trained = 0.0

    # Training shifts each marginal upward (capped at 0.99). RTSR-specific uplift
    # applies to B1 (resolved beliefs); general uplift to B2/B3.
    def shift(p, up):
        return float(np.clip(p * (1 + up), 0, 0.99))

    p1_t = shift(p1, uplift_rtsr)
    p2_t = shift(p2, uplift_general)
    p3_t = shift(p3, uplift_general)

    # Joint distribution over count of beliefs held, assuming independence
    # (belief_corr is a placeholder hook for a future copula adjustment).
    def tier_shares(pa, pb, pc):
        # P(k of 3) for independent Bernoullis
        ps = [pa, pb, pc]
        share = {0: 1.0, 1: 0.0, 2: 0.0, 3: 0.0}
        # enumerate all 2^3 outcomes
        share = {0:0, 1:0, 2:0, 3:0}
        for mask in range(8):
            prob = 1.0
            k = 0
            for i, p in enumerate(ps):
                bit = (mask >> i) & 1
                prob *= p if bit else (1 - p)
                k += bit
            share[k] += prob
        return share

    # Population-weighted mix: trained fraction uses uplifted ps; rest baseline.
    s_trained  = tier_shares(p1_t, p2_t, p3_t)
    s_baseline = tier_shares(p1,   p2,   p3)
    mix = {k: frac_trained * s_trained[k] + (1 - frac_trained) * s_baseline[k]
           for k in (0, 1, 2, 3)}

    # Adverse event rates per 100k by tier (multipliers of Army baseline)
    base_suicide = DF["suicide_rate_per_100k"] / 100_000
    base_ideation = DF["ideation_or_attempt_rate_per_100k"] / 100_000
    # tier 3 = resilient (0.25x baseline); tier 2 = baseline; tier 0/1 = high mult
    tier_suicide_mult  = {3: 0.25, 2: 1.0, 1: high_risk_suicide_mult, 0: high_risk_suicide_mult * 1.5}
    tier_attempt_mult  = {3: 0.25, 2: 1.0, 1: high_risk_suicide_mult, 0: high_risk_suicide_mult * 1.5}
    # Career attrition probability per soldier-year by tier (rough placeholder)
    tier_attrition_p   = {3: 0.02, 2: 0.06, 1: 0.15, 0: 0.20}

    expected_suicides = sum(N * mix[k] * base_suicide  * tier_suicide_mult[k] for k in mix)
    expected_attempts = sum(N * mix[k] * base_ideation * tier_attempt_mult[k]  for k in mix)
    expected_separations = sum(N * mix[k] * tier_attrition_p[k] for k in mix)

    adverse_cost = (expected_suicides    * cost_per_suicide
                  + expected_attempts    * cost_per_attempt
                  + expected_separations * cost_per_separation)

    return prog_cost + adverse_cost


# Baseline gap: FULL - PARTIAL (negative => full funding is cheaper overall)
def gap(**kw):
    return expected_cost("full", **kw) - expected_cost("partial", **kw)

baseline_gap = gap()

# Sensitivity ranges for each uncertainty
ranges = {
    "Training uplift (RTSR-specific)":   ("uplift_rtsr",            0.10, 0.80),
    "Training uplift (general)":         ("uplift_general",         0.10, 0.50),
    "High-risk suicide multiplier":      ("high_risk_suicide_mult", 1.5,  8.0),
    "Cost per suicide ($)":              ("cost_per_suicide",       3_000_000, 15_000_000),
    "Cost per attempt ($)":              ("cost_per_attempt",       10_000,    150_000),
    "Cost per separation ($)":           ("cost_per_separation",    30_000,    200_000),
}

rows = []
for label, (kwarg, lo, hi) in ranges.items():
    g_lo = gap(**{kwarg: lo})
    g_hi = gap(**{kwarg: hi})
    swing = abs(g_hi - g_lo)
    rows.append({
        "Uncertainty": label,
        "Gap @ low":  g_lo,
        "Gap @ high": g_hi,
        "Decision swing ($)": swing,
    })

voi = pd.DataFrame(rows).sort_values("Decision swing ($)", ascending=False).reset_index(drop=True)
display(Markdown(f"**Baseline expected-cost gap (FULL − PARTIAL):** `${baseline_gap:,.0f}`"))
display(Markdown("**A negative gap means FULL funding has a *lower* expected total cost than PARTIAL.**"))
display(voi.style.format({"Gap @ low": "${:,.0f}", "Gap @ high": "${:,.0f}", "Decision swing ($)": "${:,.0f}"}))


In [ ]:
# ── VoI BAR CHART ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))
voi_sorted = voi.sort_values("Decision swing ($)", ascending=True)
ax.barh(voi_sorted["Uncertainty"], voi_sorted["Decision swing ($)"] / 1e6, color="#3b7ddd")
ax.set_xlabel("Decision swing in expected-cost gap ($M)")
ax.set_title("Value of Information — RTSR FY2027 funding decision")
ax.grid(axis="x", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()
display(Markdown(
    "**Read this chart as:** the longer the bar, the more the answer to *should we fully "
    "fund RTSR?* depends on that input. Measure the top bars first."
))


---
## Section 6 — Step 4: Calibrated Priors (Two Paths)

AIE requires **calibrated probability estimates** — not guesses.
For RTSR our three core uncertainties are the prevalences `p_B1, p_B2, p_B3`.

- **Path A — Expert calibrated**: SMEs give a 90% CI for each prevalence; we
  fit a Beta distribution to that CI.
- **Path B — Survey-data prior**: we treat the n=4,000 baseline survey as data
  and form a posterior Beta from a weak Beta(1,1) prior.

Both yield Beta distributions on each `p_Bi` that we will hand to PyMC in Section 7.


In [ ]:
# ── PATH A: Expert Calibrated 90% CI on each belief prevalence ─────────────────
# Calibrated SMEs (chaplains, behavioral-health officers) provide 90% CI per belief.
# We fit a Beta(a, b) by matching its 5th and 95th percentiles to the CI.

from scipy.stats import beta as beta_dist
from scipy.optimize import brentq

def fit_beta_to_ci(lo: float, hi: float, mean_hint: float):
    """Find Beta(a, b) whose 5th, 95th percentiles match (lo, hi)."""
    def loss(a):
        b = a * (1 - mean_hint) / mean_hint
        return (beta_dist.ppf(0.05, a, b) - lo)**2 + (beta_dist.ppf(0.95, a, b) - hi)**2
    # Search a in (0.5, 500) for the minimum via golden section
    from scipy.optimize import minimize_scalar
    res = minimize_scalar(loss, bounds=(0.5, 500), method="bounded")
    a = res.x
    b = a * (1 - mean_hint) / mean_hint
    return float(a), float(b)

# 90% CIs supplied by SMEs (placeholder values — replace with calibrated estimates)
expert_ci = {
    "B1 — resolved beliefs":          {"ci": (0.30, 0.50), "mean": 0.40},
    "B2 — cause greater than self":   {"ci": (0.50, 0.70), "mean": 0.60},
    "B3 — higher power":              {"ci": (0.65, 0.85), "mean": 0.75},
}

expert_priors = {}
for name, info in expert_ci.items():
    a, b = fit_beta_to_ci(info["ci"][0], info["ci"][1], info["mean"])
    expert_priors[name] = {"a": a, "b": b, "ci": info["ci"], "mean": a / (a + b)}

display(Markdown("**Path A — Expert-calibrated Beta priors:**"))
display(pd.DataFrame(expert_priors).T.style.format({"a": "{:.2f}", "b": "{:.2f}", "mean": "{:.3f}"}))


In [ ]:
# ── PATH B: Survey-data Beta posterior (weak Beta(1,1) prior + n=4000 survey) ──
# We treat the baseline survey as data and form a posterior on each p_Bi.

N_SURVEY = decision_frame["target_population"]  # 4000
survey_counts = {
    "B1 — resolved beliefs":        round(decision_frame["p_belief_resolved_baseline"] * N_SURVEY),
    "B2 — cause greater than self": round(decision_frame["p_belief_cause_baseline"]    * N_SURVEY),
    "B3 — higher power":            round(decision_frame["p_higher_power_baseline"]    * N_SURVEY),
}

data_priors = {}
for name, k in survey_counts.items():
    a_post = 1 + k
    b_post = 1 + (N_SURVEY - k)
    data_priors[name] = {
        "a": a_post,
        "b": b_post,
        "mean": a_post / (a_post + b_post),
        "k": k,
        "n": N_SURVEY,
    }

display(Markdown("**Path B — Survey-derived Beta posteriors (n = 4,000):**"))
display(pd.DataFrame(data_priors).T.style.format({"a": "{:.0f}", "b": "{:.0f}", "mean": "{:.3f}"}))


In [ ]:
# ── PRIOR COMPARISON PLOT ──────────────────────────────────────────────────────
x = np.linspace(0, 1, 500)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, (name, exp) in zip(axes, expert_priors.items()):
    dat = data_priors[name]
    ax.plot(x, beta_dist.pdf(x, exp["a"], exp["b"]), label=f"Expert  Beta({exp['a']:.1f},{exp['b']:.1f})", lw=2)
    ax.plot(x, beta_dist.pdf(x, dat["a"], dat["b"]), label=f"Survey  Beta({dat['a']:.0f},{dat['b']:.0f})", lw=2, ls="--")
    ax.axvline(exp["mean"], color="C0", alpha=0.4, ls=":")
    ax.axvline(dat["mean"], color="C1", alpha=0.4, ls=":")
    ax.set_title(name)
    ax.set_xlabel("p (prevalence)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("density")
plt.tight_layout()
plt.show()
display(Markdown(
    "Survey priors are far tighter (n=4,000) than expert priors. We will use the survey "
    "posterior in Section 7 and add training-uplift uncertainty as the dominant unknown."
))


---
## Section 7 — Step 5: Bayesian Update (PyMC Model)

We build a PyMC model with:
- **Beta(1,1) priors** on the three baseline prevalences `p_B1, p_B2, p_B3`.
- **Bernoulli/Binomial likelihoods** anchored on the n=4,000 baseline survey counts.
- A **calibrated Normal prior** on the RTSR-specific *training uplift* (90% CI ≈ 30–70%).
- A **calibrated Normal prior** on the general 'seek-help' uplift (90% CI ≈ 15–45%).

From these we derive the population-level **post-training prevalences** and the
**risk-tier shares** (assuming belief independence — flagged for VoI item #1).


In [ ]:
# ── PyMC BAYESIAN MODEL — RTSR Beta-Binomial + uplift ──────────────────────────
import pymc as pm
import arviz as az

N = decision_frame["target_population"]
k1 = round(decision_frame["p_belief_resolved_baseline"] * N)
k2 = round(decision_frame["p_belief_cause_baseline"]    * N)
k3 = round(decision_frame["p_higher_power_baseline"]    * N)

with pm.Model() as rtsr_model:
    # Baseline belief prevalences
    p1 = pm.Beta("p1_resolved",    alpha=1, beta=1)
    p2 = pm.Beta("p2_cause",       alpha=1, beta=1)
    p3 = pm.Beta("p3_higher_pow",  alpha=1, beta=1)

    # Survey likelihood (binomial)
    pm.Binomial("y1", n=N, p=p1, observed=k1)
    pm.Binomial("y2", n=N, p=p2, observed=k2)
    pm.Binomial("y3", n=N, p=p3, observed=k3)

    # Training uplift priors (90% CI -> mean and sd via 1.645)
    # RTSR-specific: 0.50 +/- 0.20 -> normal(0.50, 0.122)
    uplift_rtsr    = pm.TruncatedNormal("uplift_rtsr",    mu=0.50, sigma=0.122, lower=0.0, upper=1.5)
    # General:        0.30 +/- 0.15 -> normal(0.30, 0.091)
    uplift_general = pm.TruncatedNormal("uplift_general", mu=0.30, sigma=0.091, lower=0.0, upper=1.0)

    # Post-training prevalences (deterministic). RTSR uplift -> B1; general -> B2, B3.
    p1_t = pm.Deterministic("p1_trained", pm.math.clip(p1 * (1 + uplift_rtsr),    0.0, 0.99))
    p2_t = pm.Deterministic("p2_trained", pm.math.clip(p2 * (1 + uplift_general), 0.0, 0.99))
    p3_t = pm.Deterministic("p3_trained", pm.math.clip(p3 * (1 + uplift_general), 0.0, 0.99))

    # Tier shares assuming belief independence
    pm.Deterministic("share_3of3_trained",
                     p1_t * p2_t * p3_t)
    pm.Deterministic("share_0of3_trained",
                     (1 - p1_t) * (1 - p2_t) * (1 - p3_t))
    pm.Deterministic("share_3of3_baseline",
                     p1 * p2 * p3)
    pm.Deterministic("share_0of3_baseline",
                     (1 - p1) * (1 - p2) * (1 - p3))

    trace = pm.sample(2000, tune=1000, chains=2, target_accept=0.9,
                      progressbar=False, random_seed=42)

display(Markdown("**PyMC sampling complete.**"))


In [ ]:
# ── POSTERIOR SUMMARY ──────────────────────────────────────────────────────────
summary = az.summary(
    trace,
    var_names=[
        "p1_resolved", "p2_cause", "p3_higher_pow",
        "uplift_rtsr", "uplift_general",
        "p1_trained", "p2_trained", "p3_trained",
        "share_3of3_trained", "share_3of3_baseline",
        "share_0of3_trained", "share_0of3_baseline",
    ],
    hdi_prob=0.9,
)
display(Markdown("**Posterior summary (90% HDI):**"))
display(summary)

# Quick visual: shift in 'all-3-of-3' resilient share (trained vs baseline cohort)
fig, ax = plt.subplots(figsize=(8, 4))
az.plot_posterior(trace, var_names=["share_3of3_baseline", "share_3of3_trained"],
                  hdi_prob=0.9, ax=[ax, ax])  # overlapping
ax.set_title("Posterior: share of soldiers holding all 3 resilience beliefs")
plt.tight_layout()
plt.show()


---
## Section 8 — Step 5b: Chained Bayesian DAG Visualization

The JSON from Section 4 defines a **Directed Acyclic Graph (DAG)** — a chain of dependencies  
from raw measurements → derived quantities → cost outputs.

This visualization shows exactly **where each piece of information enters the model**.

In [ ]:
# ── DAG VISUALIZATION FROM JSON MODEL ─────────────────────────────────────────
# Build NetworkX graph from the LLM-generated JSON

model_json = bayesian_model_json if "bayesian_model_json" in dir() else OFFLINE_JSON

G = nx.DiGraph()

# Color coding by node role
role_colors = {
    "prior":            "#3498db",   # blue
    "likelihood":       "#e67e22",   # orange
    "posterior":        "#2ecc71",   # green
    "decision_variable":"#9b59b6",   # purple
    "cost_node":        "#e74c3c",   # red
}

nodes      = model_json["measurement_nodes"]
node_names = {n["name"] for n in nodes}

for node in nodes:
    G.add_node(node["name"],
               role=node["role"],
               label=node["name"].replace("_", "\n"))

for node in nodes:
    for dep in node.get("depends_on", []):
        if dep in node_names:
            G.add_edge(dep, node["name"])

# Layout: hierarchical top-down by topological generation (pure-Python, no Graphviz needed)
def hierarchical_layout(graph: nx.DiGraph) -> dict:
    """Top-down layered layout based on longest path from any root."""
    depth = {}
    for n in nx.topological_sort(graph):
        preds = list(graph.predecessors(n))
        depth[n] = 0 if not preds else max(depth[p] for p in preds) + 1
    layers: dict[int, list[str]] = {}
    for node, d in depth.items():
        layers.setdefault(d, []).append(node)
    pos = {}
    max_width = max(len(row) for row in layers.values())
    for d, row in layers.items():
        row_sorted = sorted(row)
        for i, node in enumerate(row_sorted):
            x = (i + 1) / (len(row_sorted) + 1) * max_width
            y = -d  # negative so root is on top
            pos[node] = (x, y)
    return pos

try:
    pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
except (ImportError, Exception):
    pos = hierarchical_layout(G)

colors = [role_colors.get(G.nodes[n]["role"], "#95a5a6") for n in G.nodes]
labels = {n: G.nodes[n]["label"] for n in G.nodes}

fig, ax = plt.subplots(figsize=(14, 6))
nx.draw_networkx(
    G, pos=pos, ax=ax,
    labels=labels,
    node_color=colors,
    node_size=2200,
    font_size=7,
    font_color="white",
    font_weight="bold",
    edge_color="#555",
    arrows=True,
    arrowsize=18,
    width=2,
)

# Legend
legend_patches = [mpatches.Patch(color=v, label=k.replace("_", " ").title())
                  for k, v in role_colors.items()]
ax.legend(handles=legend_patches, loc="upper left", fontsize=8)
ax.set_title("Bayesian Measurement DAG — Generated from Problem Description",
             fontsize=12, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

# Print chain
chain = model_json["bayesian_update_chain"]
display(Markdown("**Bayesian Update Chain:** " + " → ".join(f"`{n}`" for n in chain)))

---
## Section 9 — Step 7: Monte Carlo Simulation & Decision Recommendation

We propagate the PyMC posteriors through the cost model for each funding option:

- **FULL** ($150K): 100% of soldiers receive the trained-prevalence draws.
- **PARTIAL** ($50K): 33% trained, 67% baseline.
- **NONE**: 0% trained.

For each draw we compute expected suicides, attempts, and separations across the
4,000-soldier cohort using tier-specific multipliers, then add the program cost.


In [ ]:
# ── MONTE CARLO COST PROPAGATION (RTSR) ────────────────────────────────────────
# Use posterior draws of (p1, p2, p3, uplifts) to propagate cost uncertainty
# for FULL / PARTIAL / NONE.

post = trace.posterior
# Flatten chain x draw into a single sample axis
def flat(name):
    return post[name].values.reshape(-1)

p1_b = flat("p1_resolved")
p2_b = flat("p2_cause")
p3_b = flat("p3_higher_pow")
p1_t = flat("p1_trained")
p2_t = flat("p2_trained")
p3_t = flat("p3_trained")
S = p1_b.size  # number of posterior draws

N_POP = decision_frame["target_population"]

# Cost & risk-multiplier assumptions (calibrated placeholders)
COST_PER_SUICIDE     = 11_000_000   # VSL anchor
COST_PER_ATTEMPT     =     50_000   # hospitalization + lost duty
COST_PER_SEPARATION  =     85_000   # recruitment + initial training replacement
BASE_SUICIDE  = decision_frame["suicide_rate_per_100k"]            / 100_000
BASE_IDEATION = decision_frame["ideation_or_attempt_rate_per_100k"] / 100_000
TIER_SUICIDE_MULT  = np.array([6.0, 4.0, 1.0, 0.25])  # tier 0,1,2,3
TIER_ATTEMPT_MULT  = np.array([6.0, 4.0, 1.0, 0.25])
TIER_ATTRITION_P   = np.array([0.20, 0.15, 0.06, 0.02])

def tier_share_array(pa, pb, pc):
    """Vectorized P(k of 3 beliefs) for k=0..3 over S posterior draws."""
    # 8 outcomes × S draws → reduce by k
    out = np.zeros((4, pa.size))
    for mask in range(8):
        b1 = (mask >> 0) & 1
        b2 = (mask >> 1) & 1
        b3 = (mask >> 2) & 1
        prob = (pa if b1 else 1 - pa) * (pb if b2 else 1 - pb) * (pc if b3 else 1 - pc)
        out[b1 + b2 + b3] += prob
    return out  # shape (4, S), order k=0,1,2,3

shares_baseline = tier_share_array(p1_b, p2_b, p3_b)   # (4, S)
shares_trained  = tier_share_array(p1_t, p2_t, p3_t)   # (4, S)

def cost_samples(option: str, rng=np.random.default_rng(7)):
    if option == "full":
        prog = decision_frame["program_cost_full_usd"]
        frac = 1.0
    elif option == "partial":
        prog = decision_frame["program_cost_partial_usd"]
        frac = decision_frame["fraction_trained_partial"]
    else:
        prog = 0.0
        frac = 0.0

    mix = frac * shares_trained + (1 - frac) * shares_baseline   # (4, S)

    # Expected event counts per draw
    exp_suicides    = N_POP * (mix * TIER_SUICIDE_MULT[:, None]).sum(axis=0) * BASE_SUICIDE
    exp_attempts    = N_POP * (mix * TIER_ATTEMPT_MULT[:, None]).sum(axis=0) * BASE_IDEATION
    exp_separations = N_POP * (mix * TIER_ATTRITION_P[:, None]).sum(axis=0)

    # Add Poisson sampling noise on the realized counts
    suicides    = rng.poisson(exp_suicides)
    attempts    = rng.poisson(exp_attempts)
    separations = rng.poisson(exp_separations)

    cost = (prog
            + suicides    * COST_PER_SUICIDE
            + attempts    * COST_PER_ATTEMPT
            + separations * COST_PER_SEPARATION)
    return cost, suicides, attempts, separations

results = {}
for opt in ("full", "partial", "none"):
    c, s, a, sep = cost_samples(opt)
    results[opt] = {
        "cost": c, "suicides": s, "attempts": a, "separations": sep,
    }

summary_rows = []
for opt, d in results.items():
    summary_rows.append({
        "Strategy": opt.upper(),
        "Mean Cost ($)":   d["cost"].mean(),
        "Median ($)":      np.median(d["cost"]),
        "P10 ($)":         np.percentile(d["cost"], 10),
        "P90 ($)":         np.percentile(d["cost"], 90),
        "E[suicides]":     d["suicides"].mean(),
        "E[attempts]":     d["attempts"].mean(),
        "E[separations]":  d["separations"].mean(),
    })

cost_df = pd.DataFrame(summary_rows).set_index("Strategy")
# Probability that FULL is cheaper than each alternative
p_full_better_partial = float((results["full"]["cost"] < results["partial"]["cost"]).mean())
p_full_better_none    = float((results["full"]["cost"] < results["none"]["cost"]).mean())

dollar_cols = ["Mean Cost ($)", "Median ($)", "P10 ($)", "P90 ($)"]
fmt = cost_df.copy()
for col in dollar_cols:
    fmt[col] = fmt[col].map(lambda v: f"${v:,.0f}")
for col in ("E[suicides]", "E[attempts]", "E[separations]"):
    fmt[col] = fmt[col].map(lambda v: f"{v:.2f}")
display(Markdown("### Monte Carlo Cost Summary — RTSR FY2027"))
display(fmt)
display(Markdown(
    f"- **P(FULL cheaper than PARTIAL):** {p_full_better_partial:.1%}\n"
    f"- **P(FULL cheaper than NONE):**    {p_full_better_none:.1%}"
))


In [ ]:
# ── DECISION VISUALIZATION ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) Cost distributions
for opt, color in zip(("full", "partial", "none"), ("#2c9f2c", "#f0a000", "#d04040")):
    axes[0].hist(results[opt]["cost"] / 1e6, bins=60, alpha=0.45,
                 label=opt.upper(), color=color, density=True)
axes[0].set_xlabel("Total expected cost ($M)")
axes[0].set_ylabel("density")
axes[0].set_title("Total cost distributions by funding option")
axes[0].legend()
axes[0].grid(linestyle=":", alpha=0.5)

# (b) Suicides avoided vs NONE
none_s = results["none"]["suicides"]
avoided_full    = none_s - results["full"]["suicides"]
avoided_partial = none_s - results["partial"]["suicides"]
axes[1].hist(avoided_full,    bins=range(int(min(avoided_partial.min(), avoided_full.min())), int(max(avoided_full.max(), avoided_partial.max())) + 2),
             alpha=0.55, color="#2c9f2c", label="FULL")
axes[1].hist(avoided_partial, bins=range(int(min(avoided_partial.min(), avoided_full.min())), int(max(avoided_full.max(), avoided_partial.max())) + 2),
             alpha=0.55, color="#f0a000", label="PARTIAL")
axes[1].set_xlabel("Suicides avoided vs NONE (per 4,000-soldier cohort)")
axes[1].set_ylabel("posterior draws")
axes[1].set_title("Adverse-event reductions")
axes[1].legend()
axes[1].grid(linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

display(Markdown(
    "### Recommendation logic\n"
    "If `P(FULL cheaper than PARTIAL)` > 0.5 AND the **expected suicides avoided** is "
    "non-trivial, the AIE recommendation is to fully fund the FY2027 program. "
    "Sensitivity is dominated by the inputs ranked at the top of the Section 5 VoI chart, "
    "so spend measurement effort there before the program executes."
))


---
## Section 10 — Your Turn: Adapt to Another Human-Capital Decision

Replace the decision frame below with **another non-equipment decision** where
an LLM-assisted Bayesian model could help — for example:

1. Funding level for a Sexual Assault Prevention training cycle.
2. Funding level for a substance-use intervention pilot.
3. A chaplaincy staffing decision across N installations.
4. A retention-bonus targeting decision for at-risk MOSs.

The same workflow applies: structure the frame, let the LLM parse it, rank VoI,
elicit calibrated priors, run a Bayesian update, propagate via Monte Carlo.

---
### ## YOUR TURN — Edit the cell below


In [ ]:
# ── PARTICIPANT EXERCISE ───────────────────────────────────────────────────────
# Replace the values below with your own human-capital / program decision.
# Then re-run ALL cells from Section 3 onward.

my_decision_frame = {
    "problem_description": """
        << DESCRIBE YOUR PROGRAM-FUNDING OR HUMAN-CAPITAL DECISION HERE >>
        Include: target population, baseline outcome rates, intervention cost,
        intervention effect size (point + 90% CI if you have it), adverse-event
        rates avoided, time horizon, and the funding levels you are choosing among.
    """,
    "decision_options": [
        "Option A — full funding",
        "Option B — partial funding",
        "Option C — no funding / status quo",
    ],
    "objective": "Minimize expected total cost (program + adverse-event-related) over the planning horizon",
    "time_horizon_months": 12,
    "target_population": 0,                  # e.g., soldiers in cohort
    "program_cost_full_usd": 0,
    "program_cost_partial_usd": 0,
    "fraction_trained_partial": 0.0,
    "baseline_outcome_rate_per_100k": 0.0,   # adverse event rate without intervention
    "intervention_uplift_point": 0.0,        # e.g., 0.30 for +30%
    "intervention_uplift_ci_90": [0.0, 0.0], # calibrated lower/upper
}

# Uncomment to use your frame and re-run from Section 3:
# decision_frame = my_decision_frame
# bayesian_model_json = parse_problem_to_json(my_decision_frame["problem_description"])
# print(json.dumps(bayesian_model_json, indent=2))

print("Exercise template ready. Fill in the fields above and uncomment the last lines.")


---
## References

1. **Hubbard, D.** (2014). *How to Measure Anything: Finding the Value of Intangibles in Business* (3rd ed.). Wiley.
2. **Hubbard, D.** (2025). *How to Measure Anything in AI: Quantitative Techniques for Decision-Making*. LinkedIn Learning.
3. **LLM-BI**: *Towards Fully Automated Bayesian Inference with Large Language Models*. arXiv:2508.08300 (Aug 2025).
4. **Lorenz & Fritz** (2026). *Scalable Delphi: LLM-Assisted Expert Elicitation for Calibrated Priors*. arXiv (Feb 2026).
5. **Salvatier, J., Wiecki, T., Fonnesbeck, C.** (2016). Probabilistic programming in Python using PyMC3. *PeerJ Computer Science*.
6. **Kumar & Klefsjö** (1994). Proportional hazards model: a review. *Reliability Engineering & System Safety*, 44(2), 177-188.

---
*Notebook version 1.0 — RAMS 2027 Tutorial Submission*  
*Author: Kirtis Christensen | Bayesian Mentor: Grok (xAI)*  
*Generated: April 30, 2026*